# A/B Test – Business Impact Analysis

## Business Scenario

An e-commerce company redesigned its checkout page (Variant B) to improve conversion rate.

We aim to evaluate:

- Whether the new design increases conversion
- Whether the uplift is statistically significant
- The estimated revenue impact
- A clear executive recommendation


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

In [2]:
np.random.seed(42)

n_control = 12000
n_variant = 12000

# Conversion rates (realistic uplift)
p_control = 0.10
p_variant = 0.112  # +1.2pp uplift

control_conv = np.random.binomial(1, p_control, n_control)
variant_conv = np.random.binomial(1, p_variant, n_variant)

ab = pd.DataFrame({
    "user_id": np.arange(n_control + n_variant),
    "group": ["control"] * n_control + ["variant"] * n_variant,
    "converted": np.concatenate([control_conv, variant_conv])
})

ab.head()


,user_id,group,converted
0,0,control,0
1,1,control,1
2,2,control,0
3,3,control,0
4,4,control,0


In [3]:
# Conversion rates
conversion_rates = ab.groupby("group")["converted"].mean()

conversion_rates


group
control    0.097667
variant    0.117917
Name: converted, dtype: float64

In [4]:
control_rate = conversion_rates["control"]
variant_rate = conversion_rates["variant"]

uplift = variant_rate - control_rate

print(f"Control Conversion Rate: {control_rate:.4f}")
print(f"Variant Conversion Rate: {variant_rate:.4f}")
print(f"Absolute Uplift: {uplift:.4f}")
print(f"Relative Uplift: {(uplift/control_rate)*100:.2f}%")


Control Conversion Rate: 0.0977
Variant Conversion Rate: 0.1179
Absolute Uplift: 0.0203
Relative Uplift: 20.73%


In [5]:
# Number of conversions
conversions = ab.groupby("group")["converted"].sum()

n_control = ab[ab["group"] == "control"].shape[0]
n_variant = ab[ab["group"] == "variant"].shape[0]

success = np.array([conversions["control"], conversions["variant"]])
nobs = np.array([n_control, n_variant])

z_stat, p_value = proportions_ztest(success, nobs)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.6f}")


Z-statistic: -5.0580
P-value: 0.000000


In [6]:
monthly_visitors = 500000
aov = 80

expected_control_orders = monthly_visitors * control_rate
expected_variant_orders = monthly_visitors * variant_rate

additional_orders = expected_variant_orders - expected_control_orders
monthly_revenue_uplift = additional_orders * aov

print(f"Expected Additional Orders per Month: {additional_orders:.0f}")
print(f"Estimated Monthly Revenue Uplift: ${monthly_revenue_uplift:,.2f}")


Expected Additional Orders per Month: 10125
Estimated Monthly Revenue Uplift: $810,000.00


## Statistical Conclusion

The A/B test shows a statistically significant improvement in conversion rate (p < 0.001).

The variant increases conversion by approximately X% relative uplift.

Given an estimated monthly traffic of 500,000 users and an average order value of $80:

- Expected additional orders per month: ~10,125
- Estimated monthly revenue uplift: ~$810,000

### Executive Recommendation

The new checkout design (Variant B) should be implemented.

The uplift is statistically significant and financially meaningful, generating substantial incremental revenue.
